# CPRI Hackathon — Person 1 Stage 6: Baseline Anomaly Detectors

This notebook demonstrates Person 1 Stage 6 baseline detectors for Task 01:
1. **Deterministic Quality Baseline**
2. **Robust Statistical Baseline (IQR / Outliers)**
3. **Z-Score Statistical Baseline**
4. **Unsupervised Baseline (Isolation Forest & LOF)**
5. **Residual / Physical Consistency Baseline**
6. **False Positive & False Negative Analysis**
7. **Operating Regime Breakdown**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.dataset_loader import load_training_data, TASK01_TARGET
from src.stage5_features import create_stage5_features
from src.stage6_baselines import (
    run_deterministic_quality_baseline,
    run_iqr_statistical_baseline,
    run_zscore_statistical_baseline,
    run_unsupervised_baseline,
    run_residual_consistency_baseline,
    evaluate_detector,
    evaluate_regime_performance
)

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 1. Execute Baseline Detectors on Training Data

In [ ]:
df_train = load_training_data()
df_feat, _ = create_stage5_features(df_train, fit_normal_models=True)
y_true = (df_train[TASK01_TARGET] == 'Invalid').astype(int).values

p1, s1 = run_deterministic_quality_baseline(df_feat)
p2, s2 = run_iqr_statistical_baseline(df_feat)
p3, s3 = run_zscore_statistical_baseline(df_feat)
p4, s4 = run_unsupervised_baseline(df_feat, df_feat, method='isolation_forest')
p5, s5 = run_residual_consistency_baseline(df_feat, max_res_threshold=3.0)

eval_records = [
    evaluate_detector(y_true, p1, s1, 'Deterministic_Quality', 'Quality Flags', 'Rule'),
    evaluate_detector(y_true, p2, s2, 'IQR_Statistical', 'Physical Features', '1.5*IQR'),
    evaluate_detector(y_true, p3, s3, 'ZScore_Statistical', 'Physical Features', 'Max |Z|>3.0'),
    evaluate_detector(y_true, p4, s4, 'Isolation_Forest', 'Physical Features', 'Contamination=auto'),
    evaluate_detector(y_true, p5, s5, 'Residual_Consistency', 'Residual Features', 'max_res>3.0')
]

comp_df = pd.DataFrame(eval_records).sort_values(by='Invalid_F1', ascending=False)
display(comp_df)

## 2. Visualize Baseline F1-Score Comparison

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=comp_df, x='Method', y='Invalid_F1', palette='viridis')
plt.xticks(rotation=20, ha='right')
plt.title('Baseline Anomaly Detectors — Invalid F1 Score')
plt.tight_layout()
plt.show()